In [186]:
import mysql.connector
from datetime import datetime
import os
import cv2
import shutil
import re
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tkinter import *
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk
from copy import deepcopy
from ultralytics import YOLO
import random

def resize_keep_aspect(frame, target_w, target_h):
    """
    Resize frame to fit inside target size while preserving aspect ratio.
    Adds black padding (letterbox) if needed.
    """
    h, w = frame.shape[:2]
    scale = min(target_w / w, target_h / h)
    nw, nh = int(w * scale), int(h * scale)

    resized = cv2.resize(frame, (nw, nh))
    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)

    x = (target_w - nw) // 2
    y = (target_h - nh) // 2
    canvas[y:y+nh, x:x+nw] = resized
    return canvas

In [187]:
# Config / Globals

BASE_W, BASE_H = 640, 480
video_path = None
cap = None
current_frame = None
current_frame_number = 0


frame_boxes = {}
boxes = [] 
# per-frame undo/redo 
frame_undo_stacks = {}
frame_redo_stacks = {}


global_history_undo = []
global_history_redo = []

left_mode = None
selected_index = None
is_left_dragging = False
is_right_drawing = False
start_img_x = start_img_y = 0

zoom_factor = 1.0
display_w = int(BASE_W * zoom_factor)
display_h = int(BASE_H * zoom_factor)
img_canvas_x = 0
img_canvas_y = 0
current_imgtk = None

canvas = None
canvas_width = 900
canvas_height = 600


updating_ui = False

OUTER_MARGIN = max(BASE_W, BASE_H)
# YOLO
yolo_model = None
yolo_weights_path = "best.pt"  
processing = False

# Persisted tags file 
tags_path = None
# Track unsaved edits 
dirty = False
# SQLite DB file 
db_path = None
# YOLO (dual-model inference) 
yolo_general_model = None
yolo_custom_model = None

# General pretrained model (COCO) for common vehicles
yolo_general_weights = "yolov8n.pt"


# Change this to your trained best.pt path when available
yolo_custom_weights = "D:\Anusara\Campus Final year\Finel Project\My Proposal\V-L_ditection_Camera\Video Data\weights"  # e.g. r"C:\...\runs\detect\train\weights\best.pt"

GENERAL_KEEP = {"car","bus","truck","motorcycle","bicycle"}

AUTOLOAD_TAGS_DEFAULT = True  


# INCREMENTAL YOLO TRAINING 

EPOCHS_PER_UPDATE = 10
YOLO_IMG_SIZE = 640
PROJECT_WEIGHTS_DIR = "weights"
PROJECT_DATASET_DIR = "yolo_dataset"
PROJECT_RUNS_DIR = "runs"
#  Road lane annotations 
frame_lanes = {}      
lanes = []           
lane_selected_index = None

# Tool mode: "vehicle" (boxes) or "lane" (lines)
active_tool = "vehicle"

# Lane drawing state 
lane_point_start = None  
road_lane_type = "in_lane"  


lane_color_map = {
    "in_lane": "#0000FF",   
    "next_lane": "#FFA500", 
    "far_lane": "#FF1493",  
    "other_lane": "#800080" 
}

# Lane edit state 
lane_dragging = False
lane_drag_mode = None  
lane_drag_start = (0, 0)


# per-frame undo/redo for lanes
frame_lane_undo_stacks = {}
frame_lane_redo_stacks = {}

# Frames where lanes were manually edited 
lane_manual_frames = set()

# Total frames of currently loaded video 
total_frames_global = 0

# Total frames of current loaded video (
video_total_frames = 0

#  YOLO lane segmentation 

yolo_lane_seg_weights = "D:\Anusara\Campus Final year\Finel Project\My Proposal\V-L_ditection_Camera\Video Data\weights"  # e.g. r"D:\...\weights\lane_best.pt"
yolo_lane_seg_model = None


<>:61: SyntaxWarning: invalid escape sequence '\A'
<>:116: SyntaxWarning: invalid escape sequence '\A'
<>:61: SyntaxWarning: invalid escape sequence '\A'
<>:116: SyntaxWarning: invalid escape sequence '\A'
C:\Users\Laptop Outlet\AppData\Local\Temp\ipykernel_16688\1827223614.py:61: SyntaxWarning: invalid escape sequence '\A'
  yolo_custom_weights = "D:\Anusara\Campus Final year\Finel Project\My Proposal\V-L_ditection_Camera\Video Data\weights"  # e.g. r"C:\...\runs\detect\train\weights\best.pt"
C:\Users\Laptop Outlet\AppData\Local\Temp\ipykernel_16688\1827223614.py:116: SyntaxWarning: invalid escape sequence '\A'
  yolo_lane_seg_weights = "D:\Anusara\Campus Final year\Finel Project\My Proposal\V-L_ditection_Camera\Video Data\weights"  # e.g. r"D:\...\weights\lane_best.pt"


In [188]:
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "",
    "database": "vehicle_lane_db"
}

def init_mysql():
    """Ensure tables exist (safe)."""
    conn = mysql.connector.connect(**DB_CONFIG)
    cur = conn.cursor()
    try:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS video_tags (
                id INT AUTO_INCREMENT PRIMARY KEY,
                video_path TEXT NOT NULL,
                base_w INT,
                base_h INT,
                frame_boxes_json LONGTEXT NOT NULL,
                saved_at DATETIME DEFAULT CURRENT_TIMESTAMP
            )
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS lane_tags (
                id INT AUTO_INCREMENT PRIMARY KEY,
                video_path TEXT NOT NULL,
                base_w INT,
                base_h INT,
                frame_lanes_json LONGTEXT NOT NULL,
                saved_at DATETIME DEFAULT CURRENT_TIMESTAMP
            )
        """)

        conn.commit()   # ✅ MUST be inside init_mysql()
    finally:
        cur.close()
        conn.close()


def save_tags_to_mysql(vpath: str, base_w: int, base_h: int, frame_boxes_obj):
    """Insert a new snapshot row (keeps history)."""
    init_mysql()
    fb_json = json.dumps(frame_boxes_obj, ensure_ascii=False)

    conn = mysql.connector.connect(**DB_CONFIG)
    cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO video_tags (video_path, base_w, base_h, frame_boxes_json)
            VALUES (%s, %s, %s, %s)
        """, (vpath, base_w, base_h, fb_json))
        conn.commit()
    finally:
        cur.close()
        conn.close()


def load_latest_boxes_from_db(vpath: str):
    """Load most recent saved tags for this video from MySQL. Returns dict or None."""
    try:
        init_mysql()
        conn = mysql.connector.connect(**DB_CONFIG)
        cur = conn.cursor()
        try:
            cur.execute("""
                SELECT frame_boxes_json
                FROM video_tags
                WHERE video_path = %s
                ORDER BY saved_at DESC, id DESC
                LIMIT 1
            """, (vpath,))
            row = cur.fetchone()
            if not row:
                return None
            return json.loads(row[0])
        finally:
            cur.close()
            conn.close()
    except Exception as e:
        try:
            messagebox.showerror("DB load error", str(e))
        except Exception:
            pass
        return None


def save_lanes_to_mysql(vpath: str, base_w: int, base_h: int, frame_lanes_obj):
    """Insert lanes snapshot for a video into MySQL."""
    init_mysql()
    fl_json = json.dumps(frame_lanes_obj, ensure_ascii=False)

    conn = mysql.connector.connect(**DB_CONFIG)
    cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO lane_tags (video_path, base_w, base_h, frame_lanes_json)
            VALUES (%s, %s, %s, %s)
        """, (vpath, base_w, base_h, fl_json))
        conn.commit()
    finally:
        cur.close()
        conn.close()


def load_latest_lanes_from_db(vpath: str):
    """Load most recent saved lanes for this video from MySQL. Returns dict or None."""
    try:
        init_mysql()
        conn = mysql.connector.connect(**DB_CONFIG)
        cur = conn.cursor()
        try:
            cur.execute("""
                SELECT frame_lanes_json
                FROM lane_tags
                WHERE video_path = %s
                ORDER BY saved_at DESC, id DESC
                LIMIT 1
            """, (vpath,))
            row = cur.fetchone()
            if not row:
                return None
            return json.loads(row[0])
        finally:
            cur.close()
            conn.close()
    except Exception as e:
        try:
            messagebox.showerror("DB load error", str(e))
        except Exception:
            pass
        return None


In [189]:
def init_yolo(weights_path=None):
    """Load YOLO weights once and reuse."""
    global yolo_model, yolo_weights_path
    if weights_path:
        yolo_weights_path = weights_path
    if yolo_model is None:
        yolo_model = YOLO(yolo_weights_path)
    return yolo_model


In [190]:
def get_default_tags_path(vpath: str) -> str:
    
    base, _ = os.path.splitext(vpath)
    return base + ".tags.json"

def load_tags_if_exists(path: str):
    
    global frame_boxes, frame_undo_stacks, frame_redo_stacks
    global global_history_undo, global_history_redo, dirty

    # ---- 1) Try MySQL first ----
    if video_path:
        fb = load_latest_boxes_from_db(video_path)
        if isinstance(fb, dict) and len(fb) > 0:
            frame_boxes.clear()
            for k, v in fb.items():
                try:
                    frame_boxes[int(k)] = v
                except:
                    pass

            frame_undo_stacks = {}
            frame_redo_stacks = {}
            global_history_undo = []
            global_history_redo = []
            dirty = False
            try:
                display_frame()
            except Exception:
                pass
            messagebox.showinfo("Loaded", "Previous tags loaded from MySQL.")
            return

    # ---- 2) Fallback to JSON ----
    if not path or not os.path.exists(path):
        dirty = False
        return

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        loaded = data.get("frame_boxes", {})
        if not isinstance(loaded, dict):
            loaded = {}

        frame_boxes.clear()
        for k, v in loaded.items():
            try:
                frame_boxes[int(k)] = v
            except:
                pass

        frame_undo_stacks = {}
        frame_redo_stacks = {}
        global_history_undo = []
        global_history_redo = []
        dirty = False

        try:
            display_frame()
        except Exception:
            pass

        messagebox.showinfo("Loaded", f"Previous tags loaded:\n{path}")

    except Exception as e:
        messagebox.showerror("Load tags failed", str(e))

def mark_dirty():
    global dirty
    dirty = True


def on_close():
    
    try:
        if dirty:
            if messagebox.askyesno("Unsaved", "You have unsaved changes. Save before closing?"):
                save_tags()
    except Exception:
        pass
    root.destroy()


In [191]:
# History / Undo-Redo 

def save_global_history_snapshot():
    
    snap = {
        "frame_boxes": deepcopy(frame_boxes),
        "current_frame_number": int(current_frame_number)
    }
    
    if global_history_undo and global_history_undo[-1] == snap:
        return
    global_history_undo.append(snap)
 
    global_history_redo.clear()

def global_undo(event=None):
    global frame_boxes, current_frame_number, boxes, selected_index
    if not global_history_undo:
        return

    # push current to redo
    current_snap = {"frame_boxes": deepcopy(frame_boxes), "current_frame_number": int(current_frame_number)}
    global_history_redo.append(current_snap)

    # restore
    snap = global_history_undo.pop()
    frame_boxes.clear()
    frame_boxes.update(deepcopy(snap["frame_boxes"]))
    current_frame_number = int(snap["current_frame_number"])

    load_current_frame_boxes_from_memory()
    update_frame()

def global_redo(event=None):
    global frame_boxes, current_frame_number, boxes, selected_index
    if not global_history_redo:
        return
    current_snap = {"frame_boxes": deepcopy(frame_boxes), "current_frame_number": int(current_frame_number)}
    global_history_undo.append(current_snap)
    snap = global_history_redo.pop()
    frame_boxes.clear()
    frame_boxes.update(deepcopy(snap["frame_boxes"]))
    current_frame_number = int(snap["current_frame_number"])
    load_current_frame_boxes_from_memory()
    update_frame()

In [192]:
# Frame-level undo/redo

def get_undo_stack_for_frame(f):
    return frame_undo_stacks.setdefault(f, [])

def get_redo_stack_for_frame(f):
    return frame_redo_stacks.setdefault(f, [])

def push_frame_undo():
    """Per-frame undo snapshot."""
    global current_frame_number
    if current_frame_number is None:
        return
    stack = get_undo_stack_for_frame(current_frame_number)
    snap = deepcopy(boxes)
    if stack and stack[-1] == snap:
        return
    stack.append(snap)
    frame_redo_stacks[current_frame_number] = []

def frame_undo(event=None):
    global boxes, selected_index
    stack = get_undo_stack_for_frame(current_frame_number)
    redo = get_redo_stack_for_frame(current_frame_number)
    if not stack:
        return
    prev = stack.pop()
    redo.append(deepcopy(boxes))
    boxes[:] = deepcopy(prev)
    selected_index = None
    save_current_frame_boxes_to_memory()
    redraw_boxes()

def frame_redo(event=None):
    global boxes, selected_index
    redo = get_redo_stack_for_frame(current_frame_number)
    stack = get_undo_stack_for_frame(current_frame_number)
    if not redo:
        return
    nxt = redo.pop()
    stack.append(deepcopy(boxes))
    boxes[:] = deepcopy(nxt)
    selected_index = None
    save_current_frame_boxes_to_memory()
    redraw_boxes()

In [193]:
from copy import deepcopy

# Frame memory helpers

def save_current_frame_boxes_to_memory():
    """Save current boxes into frame_boxes dict for the current frame."""
    if current_frame_number is None:
        return
    frame_boxes[current_frame_number] = deepcopy(boxes)

def load_current_frame_boxes_from_memory():
    """Load boxes list from frame_boxes for current frame (or empty if none)."""
    global boxes, selected_index
    new = deepcopy(frame_boxes.get(current_frame_number, []))
    boxes[:] = new
    frame_undo_stacks.setdefault(current_frame_number, [])
    frame_redo_stacks.setdefault(current_frame_number, [])

    selected_index = None

# Road lane memory helpers

def propagate_lanes_forward_from(start_frame: int):
    """Overwrite future frames' lanes with lanes from start_frame until a manually-edited frame is encountered."""
    if start_frame is None:
        return
    if start_frame not in frame_lanes:
        return
    base = deepcopy(frame_lanes.get(start_frame, []))
    f = start_frame + 1
    max_key = (total_frames_global - 1) if ('total_frames_global' in globals() and total_frames_global > 0) else (max(frame_lanes.keys()) if frame_lanes else start_frame)
    while f <= max_key:
        if f in lane_manual_frames:
            break
        frame_lanes[f] = deepcopy(base)
        f += 1


def save_current_frame_lanes_to_memory(manual: bool = True):
    
    if current_frame_number is None:
        return
    frame_lanes[current_frame_number] = deepcopy(lanes)
    if manual:
        lane_manual_frames.add(current_frame_number)
        propagate_lanes_forward_from(current_frame_number)
        frame_lane_redo_stacks[current_frame_number] = []


def load_current_frame_lanes_from_memory():
    
    global lanes, lane_selected_index

    if current_frame_number is None:
        lanes[:] = []
        lane_selected_index = None
        return

    if current_frame_number in frame_lanes:
        lanes[:] = deepcopy(frame_lanes.get(current_frame_number, []))
    else:
        copied = []
        if current_frame_number > 0:
            prev = current_frame_number - 1
            while prev >= 0 and prev not in frame_lanes:
                prev -= 1
            if prev >= 0:
                copied = deepcopy(frame_lanes.get(prev, []))
        lanes[:] = copied
        frame_lanes[current_frame_number] = deepcopy(lanes)  # store, but not manual

    frame_lane_undo_stacks.setdefault(current_frame_number, [])
    frame_lane_redo_stacks.setdefault(current_frame_number, [])
    lane_selected_index = None
def push_lane_undo():
    
    st = frame_lane_undo_stacks.setdefault(current_frame_number, [])
    st.append(deepcopy(lanes))
    frame_lane_redo_stacks[current_frame_number] = []

def lane_undo():
    st = frame_lane_undo_stacks.setdefault(current_frame_number, [])
    if not st:
        return
    prev = st.pop()
    rst = frame_lane_redo_stacks.setdefault(current_frame_number, [])
    rst.append(deepcopy(lanes))
    lanes[:] = deepcopy(prev)
    save_current_frame_lanes_to_memory()
    redraw_lanes()

def lane_redo():
    rst = frame_lane_redo_stacks.setdefault(current_frame_number, [])
    if not rst:
        return
    nxt = rst.pop()
    st = frame_lane_undo_stacks.setdefault(current_frame_number, [])
    st.append(deepcopy(lanes))
    lanes[:] = deepcopy(nxt)
    save_current_frame_lanes_to_memory()
    redraw_lanes()


In [194]:
# Coordinate transforms

def update_display_size():
    global display_w, display_h
    display_w = max(1, int(BASE_W * zoom_factor))
    display_h = max(1, int(BASE_H * zoom_factor))

def canvas_to_image_coords(cx, cy):
    ix = (cx - img_canvas_x) * (BASE_W / display_w)
    iy = (cy - img_canvas_y) * (BASE_H / display_h)
    return ix, iy

def image_to_canvas_coords(ix, iy):
    cx = img_canvas_x + (ix * (display_w / BASE_W))
    cy = img_canvas_y + (iy * (display_h / BASE_H))
    return cx, cy

def clamp_img_coords_allow_outside(ix, iy):
    
    min_x = -OUTER_MARGIN
    max_x = BASE_W + OUTER_MARGIN
    min_y = -OUTER_MARGIN
    max_y = BASE_H + OUTER_MARGIN
    ix = max(min_x, min(max_x, ix))
    iy = max(min_y, min(max_y, iy))
    return ix, iy


In [195]:

# Frame Display Helpers 

def update_frame():
    
    global current_frame, current_frame_number

    if not cap:
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if total_frames > 0:
        if current_frame_number < 0:
            current_frame_number = 0
        if current_frame_number >= total_frames:
            current_frame_number = total_frames - 1

    cap.set(cv2.CAP_PROP_POS_FRAMES, current_frame_number)
    ret, frame = cap.read()
    if (not ret) or (frame is None):
        return

    frame_resized = resize_keep_aspect(frame, BASE_W, BASE_H)
    current_frame = frame_resized
    display_frame()


def display_frame():
    
    global current_imgtk, img_canvas_x, img_canvas_y, display_w, display_h

    if current_frame is None:
        if canvas:
            canvas.delete("ALL_IMAGE")
        return

    update_display_size()

    # current_frame is BGR (OpenCV). Convert to RGB for PIL
    frame_rgb = cv2.cvtColor(current_frame, cv2.COLOR_BGR2RGB)

    pil = Image.fromarray(frame_rgb).resize((display_w, display_h), Image.BILINEAR)
    current_imgtk = ImageTk.PhotoImage(pil)

    if canvas is None:
        return

    canvas.delete("ALL_IMAGE")
    cw = canvas.winfo_width() or canvas_width
    ch = canvas.winfo_height() or canvas_height
    img_canvas_x = (cw - display_w) // 2
    img_canvas_y = (ch - display_h) // 2

    canvas.create_image(img_canvas_x, img_canvas_y, anchor="nw", image=current_imgtk, tags="ALL_IMAGE")
    canvas.image = current_imgtk  # keep reference

    # sync per-frame overlays before redraw (prevents lane/box jumping)
    try:
        load_current_frame_boxes_from_memory()
    except Exception:
        pass
    try:
        load_current_frame_lanes_from_memory()
    except Exception:
        pass

    # overlays
    try:
        redraw_boxes()
    except Exception:
        pass
    try:
        redraw_lanes()
    except Exception:
        pass


In [196]:
def init_yolo_models():
    
    global yolo_general_model, yolo_custom_model

    if yolo_general_model is None:
        yolo_general_model = YOLO(yolo_general_weights)

    if yolo_custom_weights and yolo_custom_model is None:
        yolo_custom_model = YOLO(yolo_custom_weights)

    return yolo_general_model, yolo_custom_model, yolo_lane_seg_model
def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    iw = max(0, inter_x2 - inter_x1)
    ih = max(0, inter_y2 - inter_y1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    denom = area_a + area_b - inter
    return float(inter / denom) if denom > 0 else 0.0


def merge_preds(custom_preds, general_preds, iou_thr=0.55):
    
    merged = list(custom_preds)
    for gp in general_preds:
        keep = True
        for cp in custom_preds:
            if iou_xyxy(gp["xyxy"], cp["xyxy"]) >= iou_thr:
                keep = False
                break
        if keep:
            merged.append(gp)
    return merged

In [197]:
def get_trained_marker_path(vpath: str) -> str:
    
    if not vpath:
        return None
    folder = os.path.dirname(vpath)
    base = os.path.splitext(os.path.basename(vpath))[0]
    return os.path.join(folder, f"{base}__trained.marker")

def mark_video_as_trained(vpath: str):
    mp = get_trained_marker_path(vpath)
    if not mp:
        return
    try:
        with open(mp, "w", encoding="utf-8") as f:
            f.write("trained=1\n")
    except Exception:
        pass

def is_video_marked_trained(vpath: str) -> bool:
    mp = get_trained_marker_path(vpath)
    return bool(mp and os.path.exists(mp))

In [198]:
def load_video():
    global video_path, cap, current_frame_number, frame_boxes
    global tags_path, dirty
    global frame_lanes  # lanes dict
    global_history_undo.clear()
    global_history_redo.clear()

    p = filedialog.askopenfilename(
        title="Select video",
        filetypes=[("Videos","*.mp4 *.avi *.mov *.mkv"), ("All","*.*")]
    )
    if not p:
        return

    # Keep Windows path as-is for OpenCV reliability
    video_path = p
    cap = cv2.VideoCapture(video_path)
    global total_frames_global
    try:
        total_frames_global = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    except Exception:
        total_frames_global = 0

    if not cap.isOpened():
        messagebox.showerror("Error", f"Failed to open video:\n{video_path}")
        cap = None
        return


    # Cache total frames for lane propagation
    global video_total_frames
    try:
        video_total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    except Exception:
        video_total_frames = 0

    # ---- Option B: decide whether to auto-load saved tags ----
    autoload_tags = AUTOLOAD_TAGS_DEFAULT
    try:
        if is_video_marked_trained(video_path):
            autoload_tags = False
    except Exception:
        pass

    # ---- Reset memory ----
    frame_boxes.clear()
    frame_lanes.clear()
    current_frame_number = 0
    save_global_history_snapshot()

    # ---- Show first frame immediately ----
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    ret, frame = cap.read()
    if (not ret) or (frame is None):
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        ret, frame = cap.read()

    if (not ret) or (frame is None):
        messagebox.showerror(
            
        )
        return

    frame_resized = resize_keep_aspect(frame, BASE_W, BASE_H)
    global current_frame
    current_frame = frame_resized
    display_frame()

    # ---- Restore annotations (DB first, then JSON) ----
    try:
        db_boxes = load_latest_boxes_from_db(video_path)
        db_lanes = load_latest_lanes_from_db(video_path)

        if isinstance(db_boxes, dict) and len(db_boxes) > 0:
            frame_boxes.clear()
            for k, v in db_boxes.items():
                try:
                    frame_boxes[int(k)] = v
                except Exception:
                    pass

        if isinstance(db_lanes, dict) and len(db_lanes) > 0:
            frame_lanes.clear()
            for k, v in db_lanes.items():
                try:
                    frame_lanes[int(k)] = v
                except Exception:
                    pass

    except Exception as e:
        print("MySQL restore skipped:", e)

    # JSON restore (only if allowed)
    try:
        tags_path = get_default_tags_path(video_path)
        if autoload_tags:
            load_tags_if_exists(tags_path)
    except Exception as e:
        print("JSON restore skipped:", e)

    dirty = False

    # ---- Load current frame data into active lists & redraw ----
    try:
        load_current_frame_boxes_from_memory()
        load_current_frame_lanes_from_memory()
        redraw_boxes()
        redraw_lanes()
    except Exception:
        pass


In [199]:
# Drawing / interaction

def on_left_press(event):
    global lane_point_start, lanes, dirty
    if active_tool == "lane":
        global lane_point_start, lane_selected_index, lane_dragging, lane_drag_mode, lane_drag_start, dirty

        ix, iy = canvas_to_image_coords(event.x, event.y)
        ix, iy = clamp_img_coords_allow_outside(ix, iy)

        # Select/drag existing lane if not mid-draw
        if lane_point_start is None:
            picked, mode = pick_lane_hit(ix, iy, thr_seg=10, thr_ep=12)
            if picked is not None:
                lane_selected_index = picked
                lane_dragging = True
                lane_drag_mode = mode  # 'move' | 'p1' | 'p2'
                lane_drag_start = (ix, iy)
                try:
                    push_lane_undo()
                except Exception:
                    pass
                redraw_lanes()
                return

        # Two-click create
        if lane_point_start is None:
            lane_selected_index = None
            lane_dragging = False
            lane_drag_mode = None
            lane_point_start = (ix, iy)
            redraw_lanes()
        else:
            try:
                push_lane_undo()
            except Exception:
                pass
            x1, y1 = lane_point_start
            x2, y2 = ix, iy
            lane_point_start = None
            if current_frame_number not in frame_lanes:
                frame_lanes[current_frame_number] = []
            frame_lanes[current_frame_number].append([int(x1), int(y1), int(x2), int(y2), str(road_lane_type)])
            load_current_frame_lanes_from_memory()
            save_current_frame_lanes_to_memory(manual=False)
            dirty = True
            redraw_lanes()
        return

    global selected_index, left_mode, is_left_dragging, start_img_x, start_img_y, updating_ui
    save_global_history_snapshot()
    ix, iy = canvas_to_image_coords(event.x, event.y)
    ix, iy = clamp_img_coords_allow_outside(ix, iy)

    selected_index = None
    left_mode = None
    
    for i, b in enumerate(boxes):
        bx, by, bw, bh, *_ = b
        if bx <= ix <= bx + bw and by <= iy <= by + bh:
            selected_index = i
            
            updating_ui = True
            try:
                selected_type.set(boxes[i][4])
                selected_lane.set(boxes[i][5])
            except:
                pass
            updating_ui = False
            
            edge_px_img = 10 * (BASE_W / max(1, display_w))
            if abs(ix - bx) <= edge_px_img:
                left_mode = "left"
            elif abs(ix - (bx + bw)) <= edge_px_img:
                left_mode = "right"
            else:
                left_mode = "move"
            push_frame_undo()
            break
            
    is_left_dragging = True
    start_img_x, start_img_y = ix, iy
    redraw_boxes()

def on_left_motion(event):
    global lane_dragging, lane_drag_mode, lane_drag_start, lane_selected_index, dirty
    if active_tool == "lane":
        if (not lane_dragging) or (lane_selected_index is None) or (lane_drag_mode is None):
            return
        ix, iy = canvas_to_image_coords(event.x, event.y)
        ix, iy = clamp_img_coords_allow_outside(ix, iy)
        dx = ix - lane_drag_start[0]
        dy = iy - lane_drag_start[1]
        try:
            x1, y1, x2, y2, lname = lanes[lane_selected_index][:5]
        except Exception:
            return
        if lane_drag_mode == "move":
            x1 += dx; y1 += dy; x2 += dx; y2 += dy
        elif lane_drag_mode == "p1":
            x1 += dx; y1 += dy
        elif lane_drag_mode == "p2":
            x2 += dx; y2 += dy
        lanes[lane_selected_index] = [int(x1), int(y1), int(x2), int(y2), lname]
        lane_drag_start = (ix, iy)
        try:
            save_current_frame_lanes_to_memory(manual=True)
        except Exception:
            pass
        dirty = True
        redraw_lanes()
        return
    global boxes, start_img_x, start_img_y
    if not is_left_dragging or selected_index is None or left_mode is None:
        return
    ix, iy = canvas_to_image_coords(event.x, event.y)
    ix, iy = clamp_img_coords_allow_outside(ix, iy)
    bx, by, bw, bh, label, lane, depth = boxes[selected_index]
    if left_mode == "move":
        dx = ix - start_img_x
        dy = iy - start_img_y
        new_x = bx + dx
        new_y = by + dy
        new_x = max(-OUTER_MARGIN, min(BASE_W + OUTER_MARGIN - bw, new_x))
        new_y = max(-OUTER_MARGIN, min(BASE_H + OUTER_MARGIN - bh, new_y))
        boxes[selected_index][0] = new_x
        boxes[selected_index][1] = new_y
        start_img_x, start_img_y = ix, iy
    elif left_mode == "left":
        new_x = max(-OUTER_MARGIN, min(ix, bx + bw - 5))
        new_w = (bx + bw) - new_x
        if new_w > 5:
            boxes[selected_index][0] = new_x
            boxes[selected_index][2] = new_w
    elif left_mode == "right":

        new_w = max(5, min(BASE_W + OUTER_MARGIN - bx, ix - bx))
        boxes[selected_index][2] = new_w
    save_current_frame_boxes_to_memory()
    redraw_boxes()

def on_left_release(event):
    global lane_dragging, lane_drag_mode
    if active_tool == "lane":
        lane_dragging = False
        lane_drag_mode = None
        # Commit lane edit and propagate forward
        try:
            save_current_frame_lanes_to_memory(manual=True)
        except Exception:
            pass
        return
    global is_left_dragging, left_mode
    is_left_dragging = False
    left_mode = None

def on_right_press(event):
    if active_tool == "lane":
        return
    global is_right_drawing, start_img_x, start_img_y, selected_index
    save_global_history_snapshot()
    ix, iy = canvas_to_image_coords(event.x, event.y)
    ix, iy = clamp_img_coords_allow_outside(ix, iy)
    is_right_drawing = True
    start_img_x, start_img_y = ix, iy
    canvas.delete("PREVIEW")
    selected_index = None
    push_frame_undo()
    redraw_boxes()

def on_right_motion(event):
    if active_tool == "lane":
        return
    if not is_right_drawing:
        return
    ix, iy = canvas_to_image_coords(event.x, event.y)
    ix, iy = clamp_img_coords_allow_outside(ix, iy)
    x1, y1 = image_to_canvas_coords(start_img_x, start_img_y)
    x2, y2 = image_to_canvas_coords(ix, iy)
    canvas.delete("PREVIEW")
    canvas.create_rectangle(x1, y1, x2, y2, outline="yellow", width=2, dash=(4,2), tags="PREVIEW")

def on_right_release(event):
    if active_tool == "lane":
        return
    global is_right_drawing, selected_index
    if not is_right_drawing:
        return
    ix, iy = canvas_to_image_coords(event.x, event.y)
    ix, iy = clamp_img_coords_allow_outside(ix, iy)
    canvas.delete("PREVIEW")
    w = ix - start_img_x
    h = iy - start_img_y
    if abs(w) > 3 and abs(h) > 3:
        x0 = start_img_x; y0 = start_img_y
        if w < 0:
            x0 = ix; w = -w
        if h < 0:
            y0 = iy; h = -h
        depth = 1 if y0 < (BASE_H/2) else 0.5
        boxes.append([x0, y0, w, h, selected_type.get(), selected_lane.get(), depth])
        selected_index = len(boxes) - 1
    is_right_drawing = False
    save_current_frame_boxes_to_memory()
    redraw_boxes()

def redraw_boxes():
    if canvas is None:
        return
    canvas.delete("BOX")
    for idx, b in enumerate(boxes):
        bx, by, bw, bh, label, lane, depth = b
        x1, y1 = image_to_canvas_coords(bx, by)
        x2, y2 = image_to_canvas_coords(bx + bw, by + bh)
        color = "#00FFFF" if lane == "in_lane" else "#FFD369" if lane == "next_lane" else "#FF007F" if lane == "far_lane" else "#00FF00"
        thickness = max(1, int(2 * depth))
        canvas.create_rectangle(x1, y1, x2, y2, outline=color, width=thickness, tags="BOX")
        canvas.create_text(x1 + 5, y1 - 10, text=f"{label} | {lane}", fill=color, anchor="w", font=("Arial", 10, "bold"), tags="BOX")
        if selected_index == idx:
            canvas.create_rectangle(x1, y1, x2, y2, outline="white", width=2, dash=(2,2), tags="BOX")

def redraw_lanes():
    
    if canvas is None:
        return

    # Remove previous lane drawings
    canvas.delete("LANE_LINE")
    canvas.delete("LANE_POINT")

    # Draw existing lane segments
    for i, lane in enumerate(lanes):
        if len(lane) < 5:
            continue
        x1, y1, x2, y2, lname = lane[:5]
        # Convert image coords -> canvas coords
        cx1, cy1 = image_to_canvas_coords(x1, y1)
        cx2, cy2 = image_to_canvas_coords(x2, y2)

        # Use same lane colors as vehicles (fallback white)
        color = lane_color_map.get(lname, "#FFFFFF") if 'lane_color_map' in globals() else "#FFFFFF"
        canvas.create_line(cx1, cy1, cx2, cy2, fill=color, width=3, tags="LANE_LINE")
        # endpoints
        canvas.create_oval(cx1-3, cy1-3, cx1+3, cy1+3, fill=color, outline=color, tags="LANE_POINT")
        canvas.create_oval(cx2-3, cy2-3, cx2+3, cy2+3, fill=color, outline=color, tags="LANE_POINT")

    # If selecting points, show start point
    if lane_point_start is not None:
        sx, sy = lane_point_start
        csx, csy = image_to_canvas_coords(sx, sy)
        canvas.create_oval(csx-5, csy-5, csx+5, csy+5, outline="#FFFF00", width=2, tags="LANE_POINT")

def _dist_point_to_segment(px, py, x1, y1, x2, y2):
    dx = x2 - x1
    dy = y2 - y1
    if dx == 0 and dy == 0:
        return ((px - x1)**2 + (py - y1)**2) ** 0.5
    t = ((px - x1) * dx + (py - y1) * dy) / float(dx*dx + dy*dy)
    t = max(0.0, min(1.0, t))
    projx = x1 + t * dx
    projy = y1 + t * dy
    return ((px - projx)**2 + (py - projy)**2) ** 0.5

def _dist(px, py, qx, qy):
    return ((px-qx)**2 + (py-qy)**2) ** 0.5

def pick_lane_hit(ix, iy, thr_seg=10, thr_ep=12):
    
    best_idx = None
    best_d = 1e9
    for i, ln in enumerate(lanes):
        if len(ln) < 5:
            continue
        x1,y1,x2,y2,_ = ln[:5]
        d1 = _dist(ix, iy, x1, y1)
        d2 = _dist(ix, iy, x2, y2)
        if d1 <= thr_ep:
            return i, "p1"
        if d2 <= thr_ep:
            return i, "p2"
        ds = _dist_point_to_segment(ix, iy, x1,y1,x2,y2)
        if ds < best_d:
            best_d = ds
            best_idx = i
    if best_idx is not None and best_d <= thr_seg:
        return best_idx, "move"
    return None, None


In [200]:

# YOLO Custom Training 


def _write_data_yaml(out_dir: str, class_names):
    
    yaml_path = os.path.join(out_dir, "data.yaml")

    # Use forward slashes for YAML safety on Windows
    safe_path = os.path.abspath(out_dir).replace("\\", "/")

    lines = []
    lines.append(f"path: {safe_path}")
    lines.append("train: images/train")
    lines.append("val: images/val")
    lines.append("names:")
    for i, n in enumerate(class_names):
        # Use single quotes to safely keep spaces etc.
        nn = str(n).replace("'", "''")
        lines.append(f"  {i}: '{nn}'")

    with open(yaml_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    return yaml_path


def export_yolo_dataset_from_video(video_path, frame_boxes, out_dir,
                                   class_names,
                                   img_size=(640,480),
                                   train_ratio=0.85,
                                   max_frames=None):

    W, H = img_size
    os.makedirs(out_dir, exist_ok=True)

    images_train = os.path.join(out_dir, "images", "train")
    images_val   = os.path.join(out_dir, "images", "val")
    labels_train = os.path.join(out_dir, "labels", "train")
    labels_val   = os.path.join(out_dir, "labels", "val")
    for p in [images_train, images_val, labels_train, labels_val]:
        os.makedirs(p, exist_ok=True)

    name_to_id = {n:i for i,n in enumerate(class_names)}

    frame_ids = sorted([int(k) for k in frame_boxes.keys()])
    if max_frames:
        frame_ids = frame_ids[:max_frames]
    if not frame_ids:
        raise ValueError("No frames in frame_boxes to export. Please tag some frames first.")

    random.shuffle(frame_ids)
    split = int(len(frame_ids) * train_ratio)
    train_set = set(frame_ids[:split])

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    exported = 0
    for fid in frame_ids:
        cap.set(cv2.CAP_PROP_POS_FRAMES, fid)
        ret, frame = cap.read()
        if not ret or frame is None:
            continue

        frame = cv2.resize(frame, (W, H))

        is_train = fid in train_set
        img_dir = images_train if is_train else images_val
        lab_dir = labels_train if is_train else labels_val

        img_name = f"frame_{fid:06d}.jpg"
        lab_name = f"frame_{fid:06d}.txt"
        img_path = os.path.join(img_dir, img_name)
        lab_path = os.path.join(lab_dir, lab_name)

        cv2.imwrite(img_path, frame)

        lines = []
        for b in frame_boxes.get(fid, []):
            if not isinstance(b, (list, tuple)) or len(b) < 5:
                continue
            x1, y1, x2, y2, label = b[0], b[1], b[2], b[3], b[4]

            if label not in name_to_id:
                continue

            x1 = max(0, min(W-1, int(x1)))
            y1 = max(0, min(H-1, int(y1)))
            x2 = max(0, min(W-1, int(x2)))
            y2 = max(0, min(H-1, int(y2)))
            if x2 <= x1 or y2 <= y1:
                continue

            xc = (x1 + x2) / 2.0 / W
            yc = (y1 + y2) / 2.0 / H
            bw = (x2 - x1) / float(W)
            bh = (y2 - y1) / float(H)

            cls_id = name_to_id[label]
            lines.append(f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

        with open(lab_path, "w", encoding="utf-8") as f:
            f.write("\\n".join(lines))

        exported += 1

    cap.release()

    yaml_path = _write_data_yaml(out_dir, class_names)
    return yaml_path, exported


def train_yolo_custom_from_tags():
    
    global yolo_custom_weights, yolo_custom_model

    if not video_path or cap is None:
        messagebox.showwarning("No video", "Load a video first.")
        return
    if not frame_boxes:
        messagebox.showwarning("No tags", "No manual tags found to train.")
        return

    # IMPORTANT: These names MUST match EXACTLY what you store in each box as b[4].
    class_names = [
        "tuk tuk",
        "lane",
        "left_lane",
        "right_lane",
        "in_lane",
        "next_lane"
    ]

    out_dir = os.path.join(os.path.dirname(video_path), "yolo_dataset")
    try:
        yaml_path, exported = export_yolo_dataset_from_video(
            video_path=video_path,
            frame_boxes=frame_boxes,
            out_dir=out_dir,
            class_names=class_names,
            img_size=(BASE_W, BASE_H),
            train_ratio=0.85
        )
        print("Exported frames:", exported)
        print("Dataset YAML:", yaml_path)
    except Exception as e:
        messagebox.showerror("Export failed", str(e))
        return

    try:
        model = YOLO("yolov8n.pt")  # start from pretrained
        model.train(data=yaml_path, epochs=50, imgsz=BASE_W)

        best_pt = os.path.join(model.trainer.save_dir, "weights", "best.pt")
        if not os.path.exists(best_pt):
            messagebox.showwarning("Train done", "Training finished but best.pt not found. Check runs/detect/ folder.")
            return

        yolo_custom_weights = best_pt
        yolo_custom_model = None  # force reload next time
        # Option B: persist 'open clean after training' for this video
        mark_video_as_trained(video_path)

        messagebox.showinfo(
            "Trained",
            "YOLO custom model trained!\\n\\n"
            f"best.pt saved at:\\n{best_pt}\\n\\n"
            "Now click 'Process Video' to merge yolov8n.pt + best.pt."
        )
    except Exception as e:
        messagebox.showerror("YOLO train failed", str(e))


In [201]:


def clear_all_boxes_and_refresh():
    
    global frame_boxes, boxes, selected_index
    global frame_undo_stacks, frame_redo_stacks
    global global_history_undo, global_history_redo, dirty

    # Clear per-frame storage + current-frame working list
    frame_boxes.clear()
    boxes.clear()
    selected_index = None

    # Clear undo/redo
    frame_undo_stacks = {}
    frame_redo_stacks = {}
    global_history_undo.clear()
    global_history_redo.clear()

    dirty = False

    # Refresh canvas (remove rectangles/text)
    try:
        if canvas is not None:
            canvas.delete("BOX")
        display_frame()
    except Exception:
        pass



In [202]:

# Save / Train / Process

def save_tags():
    global tags_path, dirty

    if not video_path:
        messagebox.showwarning("No video", "Please load a video first.")
        return

    if not tags_path:
        tags_path = get_default_tags_path(video_path)

    payload = {
        "video_path": video_path,
        "base_w": BASE_W,
        "base_h": BASE_H,
        "frame_boxes": frame_boxes,
        "frame_lanes": frame_lanes
    }

    try:
        # 1) Save JSON (same folder as video)
        with open(tags_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)

        # 2) Save to MySQL (XAMPP)
        save_tags_to_mysql(video_path, BASE_W, BASE_H, frame_boxes)
        save_lanes_to_mysql(video_path, BASE_W, BASE_H, frame_lanes)

        dirty = False
        messagebox.showinfo("Saved", f"Tags saved:\n{tags_path}\n\nSaved to MySQL database.")
    except Exception as e:
        messagebox.showerror("Save failed", str(e))

def train_model():
    
    try:
        save_tags()
    except Exception as e:
        messagebox.showerror("Save failed", str(e))
        return

    try:
        best_path, exported, classes = train_yolo_incremental_from_tags()
        messagebox.showinfo(
            "Trained (Incremental)",
            f"Updated model saved:\n{best_path}\n\nExported labeled frames: {exported}\nClasses: {len(classes)}"
        )
    except Exception as e:
        messagebox.showerror("Training failed", str(e))
        return

    try:
        clear_all_boxes_and_refresh()
    except Exception:
        pass



def process_video_with_model():
    
    global processing, current_frame_number, current_frame, cap

    if cap is None or not video_path:
        messagebox.showwarning("No video", "Please load a video first.")
        return

    gen_model, custom_model, lane_seg_model = init_yolo_models()

    processing = True
    current_frame_number = 0

    def step():
        global processing, current_frame_number, current_frame

        if not processing:
            return

        cap.set(cv2.CAP_PROP_POS_FRAMES, current_frame_number)
        ret, frame = cap.read()

        if (not ret) or (frame is None):
            processing = False
            messagebox.showinfo("Done", "Processing finished.")
            return

        frame_resized = resize_keep_aspect(frame, BASE_W, BASE_H)

        # ---- 1) General model predictions (vehicles only) ----
        general_preds = []
        try:
            res_g = gen_model.predict(source=frame_resized, conf=0.25, verbose=False)[0]
            names_g = gen_model.names
            if res_g.boxes is not None and len(res_g.boxes) > 0:
                xyxy = res_g.boxes.xyxy.cpu().numpy()
                cls_ids = res_g.boxes.cls.cpu().numpy().astype(int)
                confs = res_g.boxes.conf.cpu().numpy()

                for box, cid, cf in zip(xyxy, cls_ids, confs):
                    label = names_g.get(cid, str(cid)) if isinstance(names_g, dict) else names_g[cid]
                    if label not in GENERAL_KEEP:
                        continue
                    x1, y1, x2, y2 = map(int, box)
                    general_preds.append({
                        "label": label,
                        "conf": float(cf),
                        "xyxy": (x1, y1, x2, y2),
                        "source": "general"
                    })
        except Exception as e:
            print("General YOLO error:", e)

        # ---- 2) Custom model predictions (if available) ----
        custom_preds = []
        if custom_model is not None:
            try:
                res_c = custom_model.predict(source=frame_resized, conf=0.25, verbose=False)[0]
                names_c = custom_model.names
                if res_c.boxes is not None and len(res_c.boxes) > 0:
                    xyxy = res_c.boxes.xyxy.cpu().numpy()
                    cls_ids = res_c.boxes.cls.cpu().numpy().astype(int)
                    confs = res_c.boxes.conf.cpu().numpy()

                    for box, cid, cf in zip(xyxy, cls_ids, confs):
                        label = names_c.get(cid, str(cid)) if isinstance(names_c, dict) else names_c[cid]
                        x1, y1, x2, y2 = map(int, box)
                        custom_preds.append({
                            "label": label,
                            "conf": float(cf),
                            "xyxy": (x1, y1, x2, y2),
                            "source": "custom"
                        })
            except Exception as e:
                print("Custom YOLO error:", e)

        # ---- 3) Lane segmentation predictions (mask -> polyline) ----
        lane_polys = []
        if 'lane_seg_model' in locals() and lane_seg_model is not None:
            try:
                res_l = lane_seg_model.predict(source=frame_resized, conf=0.25, verbose=False)[0]
                names_l = lane_seg_model.names
                if getattr(res_l, "masks", None) is not None and res_l.masks is not None:
                    cls_ids = res_l.boxes.cls.cpu().numpy().astype(int) if (res_l.boxes is not None and len(res_l.boxes) > 0) else []
                    polys = res_l.masks.xy
                    for j, poly in enumerate(polys):
                        cid = int(cls_ids[j]) if j < len(cls_ids) else 0
                        lname = names_l.get(cid, str(cid)) if isinstance(names_l, dict) else names_l[cid]
                        lane_polys.append((lname, poly))
            except Exception as e:
                print("Lane SEG YOLO error:", e)

        # ---- Merge (prefer custom on overlaps) ----
        merged_preds = merge_preds(custom_preds, general_preds, iou_thr=0.55)

        # ---- Draw ----
        annotated = frame_resized.copy()

        # Pre-compute lane contours for point-in-polygon test (to know which lane a vehicle is in)
        lane_contours = []
        try:
            for lname, poly in lane_polys:
                if poly and len(poly) >= 3:
                    cnt = np.array(poly, dtype=np.int32).reshape((-1, 1, 2))
                    lane_contours.append((str(lname), cnt))
        except Exception:
            lane_contours = []

        def lane_at_point(px, py):
            for lname, cnt in lane_contours:
                try:
                    if cv2.pointPolygonTest(cnt, (float(px), float(py)), False) >= 0:
                        return lname
                except Exception:
                    pass
            return None

        for p in merged_preds:
            x1, y1, x2, y2 = p["xyxy"]
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            # Lane name (from prediction if present, otherwise inferred from lane polygons)
            lane_here = p.get("lane") or p.get("lane_name") or lane_at_point(cx, cy) or "unknown"

            # RULE: truck & bus should be in 'in_lane' only -> if not, draw RED
            is_heavy = str(p.get("label", "")).lower() in {"truck", "bus"}
            color = (0, 0, 255) if (is_heavy and lane_here != "in_lane") else (0, 255, 0)

            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
            cv2.putText(
                annotated,
                f'{p["label"]} {p["conf"]:.2f} | {lane_here}',
                (x1, max(0, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                1,
                cv2.LINE_AA
            )
# ---- Draw lane segmentation masks (polylines) ----
        if lane_polys:
            for lname, poly in lane_polys:
                try:
                    import numpy as np
                    pts = np.array(poly, dtype=int)
                except Exception:
                    continue
                col_hex = lane_color_map.get(lname, "#FFFFFF")
                col_hex = col_hex.lstrip("#")
                bgr = (int(col_hex[4:6], 16), int(col_hex[2:4], 16), int(col_hex[0:2], 16))
                cv2.polylines(annotated, [pts], isClosed=True, color=bgr, thickness=3)

        current_frame = annotated
        display_frame()

        current_frame_number += 1
        root.after(1, step)

    # Start loop
    step()


def stop_processing():
    """Stop the processing loop."""
    global processing
    processing = False


In [203]:
# Dropdown changes
def on_type_change(*args):
    global selected_index
    if updating_ui: return 
    if selected_index is not None and 0 <= selected_index < len(boxes):
        save_global_history_snapshot()
        push_frame_undo()
        boxes[selected_index][4] = selected_type.get()
        save_current_frame_boxes_to_memory()
        redraw_boxes()

def on_lane_change(*args):
    global selected_index
    if updating_ui: return 
    if selected_index is not None and 0 <= selected_index < len(boxes):
        save_global_history_snapshot()
        push_frame_undo()
        boxes[selected_index][5] = selected_lane.get()
        save_current_frame_boxes_to_memory()
        redraw_boxes()

In [204]:
# Zoom

def on_mouse_wheel(event):
    global zoom_factor
    if hasattr(event, "delta"):
        zoom_factor *= 1.1 if event.delta > 0 else 1/1.1
    else:
        zoom_factor *= 1.1 if event.num == 4 else 1/1.1
    zoom_factor = max(0.2, min(zoom_factor, 5.0))
    display_frame()


In [205]:
# Frame navigation (save/load per-frame) 

def next_frame(event=None):
    global current_frame_number, selected_index, boxes, updating_ui
    if cap:
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        if current_frame_number < total - 1:
            # Save current frame state before moving
            save_current_frame_boxes_to_memory()
            save_current_frame_lanes_to_memory()
            save_global_history_snapshot()

            prev_selected = selected_index
            
            box_to_carry = None
            if prev_selected is not None and 0 <= prev_selected < len(boxes):
                box_to_carry = deepcopy(boxes[prev_selected]) 
            
            #  Calculate new index
            next_idx = current_frame_number + 1 
            next_frame_boxes = deepcopy(frame_boxes.get(next_idx, []))
            # Lanes: copy previous frame lanes forward if next frame has none
            prev_lanes = deepcopy(frame_lanes.get(current_frame_number, []))
            next_lanes = deepcopy(frame_lanes.get(next_idx, []))
            if not next_lanes and prev_lanes:
                frame_lanes[next_idx] = deepcopy(prev_lanes)

            
            new_selected_index = None 
            
            if box_to_carry is not None:
                if not next_frame_boxes:

                    next_frame_boxes.append(box_to_carry)
                    new_selected_index = 0
                else:
                    next_frame_boxes.append(box_to_carry)
                    new_selected_index = len(next_frame_boxes) - 1 
                    
                frame_boxes[next_idx] = next_frame_boxes
                
            else:
                pass
            current_frame_number = next_idx
            load_current_frame_boxes_from_memory()
            load_current_frame_lanes_from_memory()
            update_frame()

           
            if new_selected_index is not None and 0 <= new_selected_index < len(boxes):
                selected_index = new_selected_index
                updating_ui = True
                try:
                    selected_type.set(boxes[selected_index][4])
                    selected_lane.set(boxes[selected_index][5])
                except:
                    pass
                updating_ui = False
            else:
                selected_index = None

            redraw_boxes()


def prev_frame(event=None):
    global current_frame_number, selected_index
    if cap and current_frame_number > 0:
        save_current_frame_boxes_to_memory()
        save_current_frame_lanes_to_memory()
        save_global_history_snapshot()
        current_frame_number -= 1
        load_current_frame_boxes_from_memory()
        load_current_frame_lanes_from_memory()
        selected_index = None 
        update_frame()

In [206]:
 #Arrow key resize/move (rules implemented as requested)
 
MOVE_STEP = 5

def on_arrow(event):
    global boxes, selected_index
    if selected_index is None:
        return
    save_global_history_snapshot()
    push_frame_undo()

    x, y, w, h, label, lane, depth = boxes[selected_index]
    shift = (event.state & 0x0001) != 0 

# UP ARROW 
    if event.keysym == "Up":
        if not shift:
            new_y = y - MOVE_STEP
            new_h = (y + h) - new_y
            new_y = max(-OUTER_MARGIN, new_y)
            new_h = max(5, min(BASE_H + OUTER_MARGIN - new_y, new_h))
            y, h = new_y, new_h
        else:
            bottom = y + h
            new_h = max(5, h - MOVE_STEP)
            new_y = bottom - new_h
            y, h = new_y, new_h

    # DOWN ARROW
    elif event.keysym == "Down":
        if not shift:
            new_h = h + MOVE_STEP
            new_h = min(BASE_H + OUTER_MARGIN - y, new_h)
            h = max(5, new_h)
        else:
            new_h = max(5, h - MOVE_STEP)
            h = new_h

    # LEFT ARROW
    elif event.keysym == "Left":
        if not shift:
            new_x = x - MOVE_STEP
            new_w = (x + w) - new_x
            new_x = max(-OUTER_MARGIN, new_x)
            new_w = max(5, min(BASE_W + OUTER_MARGIN - new_x, new_w))
            x, w = new_x, new_w
        else:
            right = x + w
            cand_x = min(x + MOVE_STEP, right - 5)
            new_w = right - cand_x
            x, w = max(-OUTER_MARGIN, cand_x), max(5, new_w)

    # RIGHT ARROW
    elif event.keysym == "Right":
        if not shift:
            new_w = w + MOVE_STEP
            new_w = min(BASE_W + OUTER_MARGIN - x, new_w)
            w = max(5, new_w)
        else:
            new_w = max(5, w - MOVE_STEP)
            w = new_w

    boxes[selected_index] = [x, y, w, h, label, lane, depth]
    save_current_frame_boxes_to_memory()
    redraw_boxes()

In [207]:
# Delete selected box (Delete key only)

def delete_selected_box(event=None):
    global selected_index, boxes
    if selected_index is None or not (0 <= selected_index < len(boxes)):
        messagebox.showwarning("No selection", "No box selected to delete.")
        return
    ans = messagebox.askyesno("Delete box", "Are you sure you want to delete the selected box?")
    if not ans:
        return

    save_global_history_snapshot()
    push_frame_undo()
    try:

        del boxes[selected_index]
    except Exception as e:
        print("Delete error:", e)
    selected_index = None
    save_current_frame_boxes_to_memory()
    redraw_boxes()

In [208]:
def on_ctrl_z_router(event=None):
    # Ctrl+Z: lane undo in lane tool, otherwise global undo
    if active_tool == "lane":
        try:
            lane_undo()
        except Exception:
            pass
        return
    try:
        global_undo()
    except Exception:
        pass


def on_ctrl_y_router(event=None):
    # Ctrl+Y: lane redo in lane tool, otherwise global redo
    if active_tool == "lane":
        try:
            lane_redo()
        except Exception:
            pass
        return
    try:
        global_redo()
    except Exception:
        pass


In [209]:

import os, random
import cv2
import numpy as np
from ultralytics import YOLO

LANE_CLASSES = ["in_lane", "next_lane", "far_lane", "other_lane"]

def lane_line_to_thin_poly(x1, y1, x2, y2, thickness=6):
    dx, dy = x2-x1, y2-y1
    length = float((dx*dx + dy*dy) ** 0.5)
    if length < 1e-6:
        return None
    nx, ny = -dy/length, dx/length
    off = thickness/2.0
    return [(x1+nx*off, y1+ny*off),
            (x1-nx*off, y1-ny*off),
            (x2-nx*off, y2-ny*off),
            (x2+nx*off, y2+ny*off)]

def write_seg_label(txt_path, lanes_for_frame, img_w, img_h, thickness=6):
    out=[]
    for ln in lanes_for_frame:
        if len(ln) < 5: 
            continue
        x1,y1,x2,y2,lname = ln[:5]
        if lname not in LANE_CLASSES:
            lname="other_lane"
        poly = lane_line_to_thin_poly(float(x1),float(y1),float(x2),float(y2), thickness=thickness)
        if poly is None: 
            continue
        cid = LANE_CLASSES.index(lname)
        coords=[]
        for px,py in poly:
            coords += [max(0,min(1,px/img_w)), max(0,min(1,py/img_h))]
        out.append(str(cid) + " " + " ".join(f"{v:.6f}" for v in coords))
    with open(txt_path,"w",encoding="utf-8") as f:
        f.write("\n".join(out))

def export_lane_seg_dataset(video_path, frame_lanes_dict, out_dir, base_w=640, base_h=480, val_ratio=0.1, thickness=6):
    os.makedirs(out_dir, exist_ok=True)
    for p in ["images/train","images/val","labels/train","labels/val"]:
        os.makedirs(os.path.join(out_dir,p), exist_ok=True)

    frames = sorted([int(k) for k,v in frame_lanes_dict.items() if v])
    if not frames:
        raise RuntimeError("No lane annotations found.")

    random.shuffle(frames)
    n_val = max(1, int(len(frames)*val_ratio))
    val_set=set(frames[:n_val])

    cap=cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    for fn in frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, fn)
        ok, frame = cap.read()
        if (not ok) or frame is None:
            continue
        frame=cv2.resize(frame,(base_w,base_h))
        split="val" if fn in val_set else "train"
        img_path=os.path.join(out_dir,f"images/{split}/frame_{fn:06d}.jpg")
        lab_path=os.path.join(out_dir,f"labels/{split}/frame_{fn:06d}.txt")
        cv2.imwrite(img_path, frame)
        write_seg_label(lab_path, frame_lanes_dict[fn], base_w, base_h, thickness=thickness)
    cap.release()

    data_yaml=os.path.join(out_dir,"lane_data.yaml")
    yaml_txt=f"path: {out_dir.replace('\\\\','/')}\ntrain: images/train\nval: images/val\nnames:\n"
    for i,n in enumerate(LANE_CLASSES):
        yaml_txt += f"  {i}: {n}\n"
    with open(data_yaml,"w",encoding="utf-8") as f:
        f.write(yaml_txt)

    return data_yaml

def train_lane_seg(video_path, frame_lanes_dict, weights_out_dir, epochs=50, imgsz=640, thickness=6):
    global yolo_lane_seg_weights
    os.makedirs(weights_out_dir, exist_ok=True)
    ds_dir=os.path.join(weights_out_dir,"lane_seg_dataset")
    data_yaml=export_lane_seg_dataset(video_path, frame_lanes_dict, ds_dir, base_w=BASE_W, base_h=BASE_H, thickness=thickness)
    model=YOLO("yolov8n-seg.pt", task="segment")
    results=model.train(data=data_yaml, epochs=int(epochs), imgsz=int(imgsz), task="segment")
    run_dir=getattr(results,"save_dir", None)
    if run_dir is None:
        run_dir=os.path.join("runs","segment","train")
    best_src=os.path.join(str(run_dir),"weights","best.pt")
    best_dst=os.path.join(weights_out_dir,"lane_best.pt")
    if not os.path.exists(best_src):
        raise RuntimeError(f"best.pt not found: {best_src}")
    import shutil
    shutil.copy2(best_src, best_dst)
    yolo_lane_seg_weights=best_dst
    return best_dst

def train_lane_seg_from_current_tags():
    if not video_path:
        messagebox.showwarning("No video","Load a video first.")
        return
    weights_dir=os.path.join(os.path.dirname(video_path),"weights")
    try:
        best=train_lane_seg(video_path, frame_lanes, weights_dir, epochs=50, imgsz=640, thickness=6)
        messagebox.showinfo("Lane Seg Trained", f"Saved:\n{best}\n\nSet yolo_lane_seg_weights automatically.")
    except Exception as e:
        messagebox.showerror("Lane Seg Train Failed", str(e))


In [ ]:
# GUI Setup

root = Tk()
root.title("Vehicle and Lane Tagging")
root.configure(bg="#121212")


root.protocol("WM_DELETE_WINDOW", on_close)
Label(root, text="Vehicle and Lane Detection", bg="#121212", fg="#E0E0E0", font=("Arial",16,"bold")).pack(pady=10)

button_frame = Frame(root, bg="#121212")
button_frame.pack(pady=(0,12))

def styled_button(parent, text, cmd):
    return Button(parent, text=text, command=cmd, bg="#242626", fg="white", font=("Arial",11,"bold"), relief="flat", padx=10, pady=6)

styled_button(button_frame, "Load Video", load_video).grid(row=0, column=0, padx=6)
styled_button(button_frame, "Save Tags", save_tags).grid(row=0, column=1, padx=6)
styled_button(button_frame, "Train Model", train_model).grid(row=0, column=2, padx=6)
styled_button(button_frame, "Train Lane Seg", train_lane_seg_from_current_tags).grid(row=0, column=3, padx=6)
styled_button(button_frame, "Process Video", process_video_with_model).grid(row=0, column=4, padx=6)

main_frame = Frame(root, bg="#121212")
main_frame.pack(fill=BOTH, expand=True)

left_panel = Frame(main_frame, bg="#1A1A1A", padx=12, pady=12)
left_panel.pack(side=LEFT, fill=Y)

Label(left_panel, text="Vehicle Type", bg="#1A1A1A", fg="#D0D0D0", font=("Arial",12,"bold")).pack(anchor="w")
selected_type = StringVar(value="car")
selected_type.trace_add("write", on_type_change)
OptionMenu(left_panel, selected_type, "car", "van", "bus", "truck", "bike", "tuk tuk", "bicycle").pack(fill=X, pady=6)

Label(left_panel, text="Lane", bg="#1A1A1A", fg="#D0D0D0", font=("Arial",12,"bold")).pack(anchor="w", pady=(8,0))
selected_lane = StringVar(value="in_lane")
selected_lane.trace_add("write", on_lane_change)
OptionMenu(left_panel, selected_lane, "in_lane", "next_lane", "far_lane", "other_lane").pack(fill=X, pady=6)

# --- Tool selection ---
Label(left_panel, text="Tool", bg="#1A1A1A", fg="#D0D0D0", font=("Arial",12,"bold")).pack(anchor="w", pady=(14,0))

tool_frame = Frame(left_panel, bg="#1A1A1A")
tool_frame.pack(fill=X, pady=6)

def set_vehicle_tool():
    global active_tool, lane_point_start
    active_tool = "vehicle"
    lane_point_start = None
    redraw_lanes()

def set_lane_tool():
    global active_tool
    active_tool = "lane"

styled_button(tool_frame, "Vehicle", set_vehicle_tool).grid(row=0, column=0, padx=4)
styled_button(tool_frame, "Road lane", set_lane_tool).grid(row=0, column=1, padx=4)

Label(left_panel, text="Road lane type", bg="#1A1A1A", fg="#D0D0D0", font=("Arial",12,"bold")).pack(anchor="w", pady=(8,0))
selected_road_lane = StringVar(value="in_lane")
def on_road_lane_change(*_):
    global road_lane_type
    if updating_ui:
        return
    road_lane_type = selected_road_lane.get()
selected_road_lane.trace_add("write", on_road_lane_change)
OptionMenu(left_panel, selected_road_lane, "in_lane", "next_lane", "far_lane", "other_lane").pack(fill=X, pady=6)

canvas = Canvas(main_frame, width=canvas_width, height=canvas_height, bg="black", highlightthickness=2, highlightbackground="#00ADB5")
canvas.pack(side=RIGHT, fill=BOTH, expand=True, padx=12, pady=12)

# Bindings for mouse and keyboard
canvas.bind("<ButtonPress-1>", on_left_press)
canvas.bind("<B1-Motion>", on_left_motion)
canvas.bind("<ButtonRelease-1>", on_left_release)

canvas.bind("<ButtonPress-3>", on_right_press)
canvas.bind("<B3-Motion>", on_right_motion)
canvas.bind("<ButtonRelease-3>", on_right_release)

canvas.bind_all("<MouseWheel>", on_mouse_wheel)
canvas.bind_all("<Button-4>", on_mouse_wheel)
canvas.bind_all("<Button-5>", on_mouse_wheel)

# frame navigation bindings 
root.bind_all('<,>', prev_frame) 
root.bind_all('<.>', next_frame)

# Global Undo/Redo 
root.bind_all('<Control-z>', on_ctrl_z_router)
root.bind_all('<Control-y>', on_ctrl_y_router)

# Arrow key bindings 
root.bind_all("<Up>", on_arrow)
root.bind_all("<Down>", on_arrow)
root.bind_all("<Left>", on_arrow)
root.bind_all("<Right>", on_arrow)

# Delete key binding 
root.bind_all('<Delete>', delete_selected_box)

# Per-frame undo/redo 
root.bind_all('<Control-Shift-Z>', frame_undo) 
root.bind_all('<Control-Shift-Y>', frame_redo)

save_global_history_snapshot()

# Delete key binding (lane or box)
root.bind("<Delete>", delete_selected_box)

root.mainloop()


In [ ]:
import os, json, shutil, cv2
from pathlib import Path

# ====== SET YOUR WINDOWS PROJECT PATH ======
PROJECT_ROOT = r"D:\Anusara\Campus Final year\Finel Project\My Proposal\V-L_ditection_Camera\VS code project"

VIDEO_PATH   = os.path.join(PROJECT_ROOT, "video", "raw_video.mp4")           # optional
MANUAL_JSON  = os.path.join(PROJECT_ROOT, "annotations", "manual_tags.json")

DATASET_DIR  = os.path.join(PROJECT_ROOT, "dataset")
IMG_TRAIN    = os.path.join(DATASET_DIR, "images", "train")
IMG_VAL      = os.path.join(DATASET_DIR, "images", "val")
LBL_TRAIN    = os.path.join(DATASET_DIR, "labels", "train")
LBL_VAL      = os.path.join(DATASET_DIR, "labels", "val")

AUTO_LBL_TRAIN = os.path.join(PROJECT_ROOT, "autolabel", "labels_auto_train")
AUTO_LBL_VAL   = os.path.join(PROJECT_ROOT, "autolabel", "labels_auto_val")
MERGED_LBL_TRAIN = os.path.join(PROJECT_ROOT, "merged", "labels_train")
MERGED_LBL_VAL   = os.path.join(PROJECT_ROOT, "merged", "labels_val")

DATA_YAML = os.path.join(PROJECT_ROOT, "data.yaml")

# Keep class order stable. Add new vehicle types ONLY at the end.
CLASS_NAMES = ["car","bus","truck","motorcycle","bicycle","tuk_tuk","lane_solid","lane_dashed"]
cls2id = {c:i for i,c in enumerate(CLASS_NAMES)}


def ensure_dirs(*dirs):
    for d in dirs:
        os.makedirs(d, exist_ok=True)


def extract_frames(video_path, out_dir, every_n=5, max_frames=None):
    ensure_dirs(out_dir)
    cap = cv2.VideoCapture(video_path)
    i, saved = 0, 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if i % every_n == 0:
            fn = f"frame_{i:06d}.jpg"
            cv2.imwrite(os.path.join(out_dir, fn), frame)
            saved += 1
            if max_frames and saved >= max_frames:
                break
        i += 1
    cap.release()
    return saved


def xyxy_to_yolo(x1,y1,x2,y2,W,H):
    x1 = max(0, min(W-1, int(x1))); x2 = max(0, min(W-1, int(x2)))
    y1 = max(0, min(H-1, int(y1))); y2 = max(0, min(H-1, int(y2)))
    if x2 <= x1 or y2 <= y1:
        return None
    xc = ((x1+x2)/2)/W
    yc = ((y1+y2)/2)/H
    w  = (x2-x1)/W
    h  = (y2-y1)/H
    return xc,yc,w,h


def manual_json_to_yolo(json_path, images_dir, labels_out_dir):
    ensure_dirs(labels_out_dir)
    data = json.load(open(json_path, "r", encoding="utf-8"))
    frame_map = data.get("frame_boxes", data)

    def get_label(b):
        return b.get("label") or b.get("class") or b.get("name")

    def get_xyxy(b):
        if all(k in b for k in ("x1","y1","x2","y2")):
            return b["x1"], b["y1"], b["x2"], b["y2"]
        if all(k in b for k in ("left","top","right","bottom")):
            return b["left"], b["top"], b["right"], b["bottom"]
        if all(k in b for k in ("x","y","w","h")):
            return b["x"], b["y"], b["x"]+b["w"], b["y"]+b["h"]
        raise KeyError("Unknown box keys. Adjust get_xyxy().")

    count = 0
    for frame_key, boxes in frame_map.items():
        try:
            frame_idx = int(frame_key)
        except:
            digits = "".join(ch for ch in str(frame_key) if ch.isdigit())
            if not digits:
                continue
            frame_idx = int(digits)

        img_name = f"frame_{frame_idx:06d}.jpg"
        img_path = os.path.join(images_dir, img_name)
        if not os.path.exists(img_path):
            continue
        img = cv2.imread(img_path)
        if img is None:
            continue
        H,W = img.shape[:2]

        lines = []
        for b in boxes:
            label = get_label(b)
            if label not in cls2id:
                continue
            x1,y1,x2,y2 = get_xyxy(b)
            y = xyxy_to_yolo(x1,y1,x2,y2,W,H)
            if not y:
                continue
            xc,yc,w,h = y
            lines.append(f"{cls2id[label]} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        with open(os.path.join(labels_out_dir, img_name.replace(".jpg",".txt")), "w", encoding="utf-8") as f:
            f.write("\\n".join(lines))
        count += 1
    return count


def autolabel_images(weights, images_dir, labels_out_dir, conf=0.6):
    ensure_dirs(labels_out_dir)
    from ultralytics import YOLO
    model = YOLO(weights)
    for fn in sorted(os.listdir(images_dir)):
        if not fn.lower().endswith((".jpg",".jpeg",".png")):
            continue
        p = os.path.join(images_dir, fn)
        res = model.predict(source=p, conf=conf, verbose=False)[0]
        H,W = res.orig_shape
        lines = []
        if res.boxes is not None and len(res.boxes)>0:
            xyxy = res.boxes.xyxy.cpu().numpy()
            cls  = res.boxes.cls.cpu().numpy().astype(int)
            for (x1,y1,x2,y2), cid in zip(xyxy, cls):
                y = xyxy_to_yolo(x1,y1,x2,y2,W,H)
                if not y:
                    continue
                xc,yc,w,h = y
                lines.append(f"{int(cid)} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
        with open(os.path.join(labels_out_dir, fn.rsplit('.',1)[0]+'.txt'), "w", encoding="utf-8") as f:
            f.write("\\n".join(lines))


def merge_labels(manual_dir, auto_dir, merged_dir):
    ensure_dirs(merged_dir)
    manual_set = set(os.listdir(manual_dir)) if os.path.exists(manual_dir) else set()
    auto_set = set(os.listdir(auto_dir)) if os.path.exists(auto_dir) else set()
    for fn in sorted(manual_set.union(auto_set)):
        if not fn.endswith(".txt"):
            continue
        src = manual_dir if fn in manual_set else auto_dir
        shutil.copyfile(os.path.join(src, fn), os.path.join(merged_dir, fn))


def write_data_yaml(yaml_path, dataset_dir, class_names):
    ds = str(Path(dataset_dir).as_posix())
    lines = [f"path: {ds}", "train: images/train", "val: images/val", "", "names:"]
    for i, n in enumerate(class_names):
        lines.append(f"  {i}: {n}")
    with open(yaml_path, "w", encoding="utf-8") as f:
        f.write("\\n".join(lines))

print("✅ Ready. Edit PROJECT_ROOT + CLASS_NAMES, then run the steps cell below.")


✅ Ready. Edit PROJECT_ROOT + CLASS_NAMES, then run the steps cell below.


In [ ]:
def set_current_frame(frame_img):
    global current_frame
    current_frame = frame_img


In [ ]:
def get_project_root() -> str:
    if video_path:
        return os.path.dirname(video_path)
    return os.getcwd()

def get_project_weights_dir() -> str:
    d = os.path.join(get_project_root(), PROJECT_WEIGHTS_DIR)
    os.makedirs(d, exist_ok=True)
    return d

def get_project_best_pt() -> str:
    return os.path.join(get_project_weights_dir(), "best.pt")

def get_dataset_root() -> str:
    d = os.path.join(get_project_root(), PROJECT_DATASET_DIR)
    os.makedirs(d, exist_ok=True)
    return d

def ensure_dataset_structure():
    root = get_dataset_root()
    for p in [
        os.path.join(root, "images", "train"),
        os.path.join(root, "images", "val"),
        os.path.join(root, "labels", "train"),
        os.path.join(root, "labels", "val"),
    ]:
        os.makedirs(p, exist_ok=True)
    return root

In [ ]:
def _safe_video_base(vpath: str) -> str:
    b = os.path.splitext(os.path.basename(vpath))[0]
    return re.sub(r"[^a-zA-Z0-9_\-]+", "_", b)

def export_current_video_tags_to_yolo_dataset(val_every: int = 10):
    if not video_path:
        raise RuntimeError("No video loaded.")
    ensure_dataset_structure()

    base = _safe_video_base(video_path)

    labels_set = set()
    for _, blist in frame_boxes.items():
        for b in blist:
            if len(b) >= 5:
                labels_set.add(str(b[4]))
    if not labels_set:
        raise RuntimeError("No labels found in current tags.")

    dataset_root = get_dataset_root()
    classes_file = os.path.join(dataset_root, "classes.txt")
    if os.path.exists(classes_file):
        with open(classes_file, "r", encoding="utf-8") as f:
            classes = [ln.strip() for ln in f.readlines() if ln.strip()]
    else:
        classes = []
    for lab in sorted(labels_set):
        if lab not in classes:
            classes.append(lab)
    with open(classes_file, "w", encoding="utf-8") as f:
        for c in classes:
            f.write(c + "\n")

    yaml_path = os.path.join(dataset_root, "data.yaml")
    data_yaml = (
        "path: " + dataset_root.replace("\\", "/") + "\n"
        "train: images/train\n"
        "val: images/val\n"
        f"nc: {len(classes)}\n"
        f"names: {classes}\n"
    )
    with open(yaml_path, "w", encoding="utf-8") as f:
        f.write(data_yaml)

    cap2 = cv2.VideoCapture(video_path)
    if not cap2.isOpened():
        raise RuntimeError("Failed to open video for export.")

    frame_ids = []
    for k in frame_boxes.keys():
        try:
            frame_ids.append(int(k))
        except Exception:
            pass
    frame_ids = sorted(frame_ids)

    exported = 0
    for idx, fno in enumerate(frame_ids):
        blist = frame_boxes.get(fno, [])
        if not blist:
            continue

        cap2.set(cv2.CAP_PROP_POS_FRAMES, int(fno))
        ret, frame = cap2.read()
        if not ret or frame is None:
            continue

        frame = cv2.resize(frame, (BASE_W, BASE_H))

        is_val = (val_every > 0 and (idx % val_every == 0))
        split = "val" if is_val else "train"

        img_name = f"{base}_f{int(fno):06d}.jpg"
        lbl_name = f"{base}_f{int(fno):06d}.txt"

        img_out = os.path.join(dataset_root, "images", split, img_name)
        lbl_out = os.path.join(dataset_root, "labels", split, lbl_name)

        cv2.imwrite(img_out, frame)

        lines = []
        for b in blist:
            if len(b) < 5:
                continue
            bx, by, bw, bh = float(b[0]), float(b[1]), float(b[2]), float(b[3])
            lab = str(b[4])
            if lab not in classes:
                continue
            cid = classes.index(lab)

            cx = bx + (bw / 2.0)
            cy = by + (bh / 2.0)

            cx_n = max(0.0, min(1.0, cx / float(BASE_W)))
            cy_n = max(0.0, min(1.0, cy / float(BASE_H)))
            w_n  = max(0.0, min(1.0, bw / float(BASE_W)))
            h_n  = max(0.0, min(1.0, bh / float(BASE_H)))

            lines.append(f"{cid} {cx_n:.6f} {cy_n:.6f} {w_n:.6f} {h_n:.6f}")

        if lines:
            with open(lbl_out, "w", encoding="utf-8") as f:
                f.write("\n".join(lines) + "\n")
            exported += 1

    cap2.release()
    if exported == 0:
        raise RuntimeError("No labeled frames could be exported (check tags/frames).")

    return yaml_path, classes, exported

In [ ]:
def train_yolo_incremental_from_tags():
    if YOLO is None:
        raise RuntimeError("Ultralytics YOLO not available. Install: pip install ultralytics")

    yaml_path, classes, exported = export_current_video_tags_to_yolo_dataset(val_every=10)

    project_best = get_project_best_pt()
    start_weights = project_best if os.path.exists(project_best) else yolo_general_weights

    model = YOLO(start_weights)

    runs_dir = os.path.join(get_project_root(), PROJECT_RUNS_DIR)
    os.makedirs(runs_dir, exist_ok=True)

    model.train(
        data=yaml_path,
        epochs=EPOCHS_PER_UPDATE,
        imgsz=YOLO_IMG_SIZE,
        project=runs_dir,
        name="detect_train",
        exist_ok=True,
        verbose=False
    )

    save_dir = str(model.trainer.save_dir)
    best_pt = os.path.join(save_dir, "weights", "best.pt")
    if not os.path.exists(best_pt):
        raise RuntimeError("Training finished but best.pt was not found in the run folder.")

    os.makedirs(os.path.dirname(project_best), exist_ok=True)
    shutil.copy2(best_pt, project_best)

    global yolo_custom_weights, yolo_custom_model
    yolo_custom_weights = project_best
    yolo_custom_model = None

    try:
        mark_video_as_trained(video_path)
    except Exception:
        pass

    return project_best, exported, classes